Retrieval Argument Generation

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner,function_tool,trace
dotenv.load_dotenv() 

True

Create chroma client and nutrition_qna

In [2]:
chromadb_client = chromadb.PersistentClient(path="../chroma")
calroies_db = chromadb_client.get_collection(name="nutrition_db")
nutrition_db_qna = chromadb_client.get_collection(name="nutrition_qna")

Create a lookup tool

In [30]:
@function_tool
def nutrition_qna_lookup(query: str, max_results: int = 2) -> str:
    """
    Tool function to ask question regarding the nutrition.

    Args:
        query: question to ask
        max_results: maximum number of results to return
    
    Returns:
        String that contains the query and the answer related to the query


    """

    results = nutrition_db_qna.query(query_texts=[query],n_results=max_results)
    if not results["documents"][0]:
        return f" No results for the query {query} "
    final_results = []
    
    for i,doc in enumerate(results["documents"][0]):
        final_results.append(doc)
    return "Related answer to your query:\n " + "\n".join(final_results)

In [ ]:
#nutrition_qna_lookup("pregnancy",2)

"Related answer to your query: \nQuestion: What are some possible physical changes that a pregnant woman could experience in the middle months of gestation?\n        Answer: During weeks 13-27, you may see an increase in weight and feel more hunger. Backaches might occur frequently, along with leg cramps and heartburn.\n\n        This Q&A pair provides information about nutrition and health topics. \nQuestion: What health issues can make a pregnancy more challenging?\n        Answer: There are several medical conditions that can potentially increase the risk associated with Pregnancy. These include Anemia during this stage, Hypertensive Disorders related to gestation, Diabetes Mellitus coexisting with Pregnancy, Obesity during pregnancy, as well as Adolescent or Teenage pregnancies.\n\n        This Q&A pair provides information about nutrition and health topics. \nQuestion: What are some hormonal changes that take place in a mother's body during pregnancy?\n        Answer: During pregn

Create Agent with tool as nutrition_qna_lookup

In [31]:
nutrition_qna_agent = Agent(
    name="Nutrition Assistant",
    instructions= """
     you are a nutrition assistant
     you provide concise results
     if you are asked a question you will lookup related answer using nutrition_qna_lookup tool
     """,
     tools=[nutrition_qna_lookup],
)

Run the agent

In [33]:
with trace("Nutrition Assistant with Nutrition QNA RAG"):
    result = await Runner.run(
        nutrition_qna_agent,
        "What are the best meal choices for pregnant women?",
    )
    print(result.final_output)


Here are the core meal recommendations for pregnant women:

- Protein: Include reliable sources at each meal (dairy, eggs, lean meats, fish (low-mercury), legumes, tofu).
- Whole grains: Choose cereals, oats, brown rice, quinoa, and millets for sustained energy.
- Fruits and vegetables: Aim for a variety, especially leafy greens (iron and folate) and vitamin-C rich fruits to aid iron absorption.
- Dairy or fortified alternatives: Milk, yogurt, cheese or fortified plant milks for calcium and vitamin D.
- Iron and folate: Foods rich in iron (meat, beans, fortified cereals) with vitamin-C sources to boost absorption; folate-rich foods include leafy greens and legumes.
- Healthy fats: Include sources of DHA/EPA (fatty fish like salmon) in safe amounts, or prenatal DHA supplements if advised.
- Fiber and fluids: High-fiber foods (fruits, vegetables, whole grains) with plenty of water to help digestion and prevent constipation.
- Safe food practices: Avoid high-mercury fish, raw/undercooked 